# Serial Baseline · your first performance number

This is Lab 01. By the end of it you have a working serial C program that solves a real physics problem (the 2-D heat equation), you have submitted it as a batch job to Crux, you have your first animation of the solution evolving in time, and you have written the very first row of `timings.csv` — the number every parallelization lab that follows this one will try to beat.

**You will:**
1. Derive the 5-point stencil for the heat equation, straight from the PDE — see how the math becomes C code line by line.
2. Fill in **one line** of a provided `heat2D.c` skeleton (the stencil update itself). Everything else — I/O, timing, argument parsing, snapshot writer — is already written so you can focus on the physics-to-code step.
3. Write your first `heat2D.pbs` script by hand — no template hidden behind a helper. This is reveal-ladder step 1 from lab 00's Part 7: helpers still submit for you, but you write the script they submit.
4. Run it on Crux, pull the output back, and produce an **animation of your initials dissipating** — your personal artifact for the lab.
5. Measure three things at once: **performance** (MLUP/s), **I/O cost** (seconds spent writing frames), and a **science correctness check** (does the total heat stay conserved to roundoff?). Power measurement comes in lab 02.
6. Produce your first four figures: performance strip, I/O strip, conservation check, and a four-panel field snapshot at t = 0, T/4, T/2, T.

> **📚 Where to look when you're stuck**
> 
> - [**Heat equation on Wikipedia**](https://en.wikipedia.org/wiki/Heat_equation) — the PDE, the derivation, the physical intuition. Skim the "Fundamental solutions" section only if curious.
> - [**Finite difference method**](https://en.wikipedia.org/wiki/Finite_difference_method) — where the 5-point stencil comes from mathematically.
> - [**NumPy `.npy` format reference**](https://numpy.org/doc/stable/reference/generated/numpy.lib.format.html) — the on-disk binary format the C code emits for each frame.
> - [**Crux Compiling and Linking**](https://docs.alcf.anl.gov/crux/compiling-and-linking/compiling-and-linking-overview/) and [**Crux Running Jobs**](https://docs.alcf.anl.gov/crux/queueing-and-running-jobs/running-jobs/) — the ALCF pages you set up in lab 00, still the authority for `cc`, `qsub`, and `#PBS` syntax.


## How this notebook works · same surfaces as lab 00

Every code cell is prefixed with a `# [Where]` comment. Nothing has changed since lab 00 — Hub for local Python, `sshRun` / `submitJob` / `sshGet` for Crux login node and compute nodes, and one **[Terminal]** step for the ssh multiplex refresh if it has expired.


In [ ]:
# [Hub] Shared toolkit - preflight/checkpoint, sshRun/sshPut/sshGet, submitJob/waitJob,
# and the plotting primitives (applyHouseStyle, renderField, plotScaling, saveFigure).
from labHelpers import *


### Set up this lab's identity

Same pattern as lab 00 — `setupLab` records where the cluster is, which project pays, and where scratch lives, and writes a `labEnv.sh` you can `source` from a terminal. **Only edit `HPC_USER`** to your ALCF username; the class project is already filled in.


In [ ]:
# [Hub] Change HPC_USER to your ALCF username, then run this cell.
env = setupLab(
    labName    = "lab01",
    host       = "crux",
    remoteUser = os.environ.get("HPC_USER", "CHANGE_ME"),
    project    = "UIC-CS455-Sp2027",
    queue      = "debug",
    scratch    = f"/eagle/UIC-CS455-Sp2027/{os.environ.get('HPC_USER','CHANGE_ME')}",
)


### Preflight · check your environment

You already passed all of these in lab 00. Running them again takes 3 seconds and catches an expired ssh multiplex channel before you hit it mid-lab.


In [ ]:
# [Hub] Environment health check.
preflight([
    check("HPC_USER is set (not CHANGE_ME)",
          lambda: (os.environ.get('HPC_USER','CHANGE_ME') != 'CHANGE_ME',
                   os.environ.get('HPC_USER','(unset)')),
          hint="Edit the setupLab() cell above with your ALCF username and re-run it."),
    check("passwordless (multiplexed) ssh to Crux", sshReachable(),
          hint="In a Hub terminal: `ssh crux hostname`, enter your MobilePASS+ passcode ONCE. "
               "That opens an 8h multiplex channel; every cell below reuses it silently."),
    check("scheduler answers on Crux", schedulerAnswers()),
    check("lab01 scratch dir on Crux", remoteFileExists(env['HPC_LAB_DIR']),
          hint="The next code cell creates it. If preflight fails here on first run, that is "
               "expected - move on to Part 1 and re-run this cell after mkdir."),
], infoRows=[('cluster', clusterHost()), ('you', env.get('HPC_USER','?')),
             ('project', env.get('HPC_PROJECT','?')), ('lab dir', env.get('HPC_LAB_DIR','?'))])


## Part 1 · From heat equation to 5-point stencil

The [**heat equation**](https://en.wikipedia.org/wiki/Heat_equation) in 2-D is the PDE

$$\frac{\partial u}{\partial t} \;=\; \alpha \, \nabla^{2} u \;=\; \alpha \left( \frac{\partial^{2} u}{\partial x^{2}} + \frac{\partial^{2} u}{\partial y^{2}} \right)$$

where $u(x,y,t)$ is the temperature field and $\alpha$ is the diffusion coefficient (how fast heat spreads). Physical intuition: **the rate of change of temperature at a point equals a constant times the local curvature of the temperature field.** Where the field is concave-up, the point heats up; where it is concave-down, it cools. Left alone long enough, everything smooths out.

To run this on a computer, both sides need to become finite numbers.


### 1a · Discretize in space · the 5-point stencil

Put a uniform grid on the unit square with spacing $h = 1/N$. Then $u_{i,j}$ means "the value of $u$ at grid cell $(i, j)$." The [**finite difference method**](https://en.wikipedia.org/wiki/Finite_difference_method) approximates the second derivative in one direction as

$$\frac{\partial^{2} u}{\partial x^{2}} \;\approx\; \frac{u_{i-1,j} \,-\, 2\,u_{i,j} \,+\, u_{i+1,j}}{h^{2}}$$

(and similarly for $y$). Adding the two directions gives the **discrete Laplacian**, the famous 5-point stencil:

```
          u[i-1, j]
              |
  u[i, j-1] -- u[i, j] -- u[i, j+1]         nabla^2 u  ~  (sum of 4 neighbors - 4*center) / h^2
              |
          u[i+1, j]
```

That is where the name comes from: five points contribute to one update.


### 1b · Discretize in time · explicit forward Euler

The simplest way to advance $u$ in time is explicit forward Euler: take the current value, add the right-hand side times a small timestep $\Delta t$.

$$u^{\,n+1}_{i,j} \;=\; u^{\,n}_{i,j} \;+\; \alpha \, \Delta t \;\cdot\; \underbrace{\frac{u^{\,n}_{i-1,j} + u^{\,n}_{i+1,j} + u^{\,n}_{i,j-1} + u^{\,n}_{i,j+1} - 4\,u^{\,n}_{i,j}}{h^{2}}}_{\text{5-point stencil}}$$

That expression translates almost line by line to C:

```c
unew[i*N + j] = u[i*N + j]
              + alpha * dt * ( u[im*N + j] + u[ip*N + j]
                             + u[i*N + jm] + u[i*N + jp]
                             - 4.0 * u[i*N + j] ) / (h*h);
```

where `im`, `ip`, `jm`, `jp` are the neighbor indices (wrapped modulo `N` for periodic boundaries — heat that leaves the right edge re-enters on the left).

**One catch: stability.** Explicit Euler on this stencil is only stable when

$$\Delta t \;<\; \frac{h^{2}}{4\,\alpha}$$

This is the [**CFL (Courant–Friedrichs–Lewy) condition**](https://en.wikipedia.org/wiki/Courant%E2%80%93Friedrichs%E2%80%93Lewy_condition). Break it and the solution blows up — you'll get `nan` everywhere in a few steps. The provided C code uses $\Delta t = 0.25\,h^{2}/\alpha$, safely under the bound.


## Part 2 · Fill in the stencil in `heat2D.c`

You have the equation, the discretization, and the C form of the update. The next cell writes an 80%-complete `heat2D.c` to your lab folder. It handles:

- Argument parsing (`--N`, `--steps`, `--snapEvery`, `--alpha`, `--outDir`)
- The initial condition (two hot squares as placeholder initials)
- The main time loop, ping-pong buffer swap, snapshot writer
- On-the-fly conservation-check row every snapshot
- Writing `timings.csv` with the schema every later lab appends to
- Writing each snapshot as a `.npy` file the notebook can load with `numpy.load`

**Your one job** is to replace the placeholder line inside the double `for` loop with the 5-point stencil update from Part 1b. The surrounding code sets up `im`, `ip`, `jm`, `jp` for you.


In [ ]:
# [Hub] Write the heat2D.c skeleton to your lab folder.
heat2DC = '/*\n * heat2D.c - 2D heat equation, serial baseline for CS 455 lab01.\n *\n * Explicit forward-Euler in time, 5-point central-difference stencil in space,\n * periodic boundary conditions on the unit square [0,1] x [0,1].\n *\n * Emits:\n *   - stdout:            one line per checkpoint, with wall time and sum(u)\n *   - <outDir>/timings.csv    (lab,variant,N,steps,threads,ranks,wall_s,mlups,io_s)\n *   - <outDir>/science.csv    (step, time, sum_u)                   -- conservation check\n *   - <outDir>/frame_XXXX.npy (raw grid dumps for renderField in the notebook)\n *\n * Build:  cc -O3 -Wall -o heat2D heat2D.c -lm\n * Run:    ./heat2D --N 256 --steps 1000 --snapEvery 50 --outDir ./out\n *\n * You (the student) fill in ONE line - the 5-point stencil in the main loop\n * below. Everything else is provided so the lab focuses on the physics->code\n * step, not on I/O boilerplate.\n */\n#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\n#include <math.h>\n#include <time.h>\n#include <unistd.h>\n\n/* -------- .npy writer (NumPy v1 format, float64, C-contiguous, 2D) -------- */\n/* We write the tiny .npy header by hand so the C code has zero deps.        */\n/* Format ref: https://numpy.org/doc/stable/reference/generated/numpy.lib.format.html */\nstatic void writeNpy2D(const char *path, const double *data, int N) {\n    FILE *f = fopen(path, "wb");\n    if (!f) { perror(path); exit(1); }\n    /* Magic + version + header */\n    const char magic[] = "\\x93NUMPY";\n    fwrite(magic, 1, 6, f);\n    unsigned char ver[2] = {1, 0};\n    fwrite(ver, 1, 2, f);\n    char header[128];\n    int n = snprintf(header, sizeof header,\n        "{\'descr\': \'<f8\', \'fortran_order\': False, \'shape\': (%d, %d), }", N, N);\n    /* pad so total header length is multiple of 64 for alignment, ends in \\n */\n    while ((10 + n + 1) % 64 != 0) header[n++] = \' \';\n    header[n++] = \'\\n\';\n    unsigned short hlen = (unsigned short)n;\n    fwrite(&hlen, 2, 1, f);\n    fwrite(header, 1, n, f);\n    fwrite(data, sizeof(double), (size_t)N * N, f);\n    fclose(f);\n}\n\n/* Wallclock in seconds since arbitrary epoch (monotonic, high-resolution).   */\nstatic double wallSeconds(void) {\n    struct timespec ts;\n    clock_gettime(CLOCK_MONOTONIC, &ts);\n    return (double)ts.tv_sec + (double)ts.tv_nsec * 1e-9;\n}\n\n/* Initial condition: your initials as hot spots on a cool background.       */\n/* Replace the block-letter grid below with your own initials in Part 3.     */\nstatic void initInitials(double *u, int N) {\n    /* Cool ambient */\n    for (int i = 0; i < N * N; ++i) u[i] = 0.10;\n\n    /* Two 8x8 hot squares as a placeholder for initials \'X\' and \'Y\'.\n     * In Part 3 you\'ll change these coordinates to spell YOUR initials. */\n    const double hot = 1.00;\n    int cx1 = N / 3,  cy1 = N / 2;    /* placeholder for first initial */\n    int cx2 = 2*N / 3, cy2 = N / 2;   /* placeholder for second initial */\n    for (int di = -4; di < 4; ++di)\n    for (int dj = -4; dj < 4; ++dj) {\n        u[(cy1+di)*N + (cx1+dj)] = hot;\n        u[(cy2+di)*N + (cx2+dj)] = hot;\n    }\n}\n\n/* Sum of u over the grid; must be conserved to roundoff on a periodic       */\n/* domain. Any drift signals a stencil or boundary bug.                      */\nstatic double gridSum(const double *u, int N) {\n    double s = 0.0;\n    for (int i = 0; i < N * N; ++i) s += u[i];\n    return s;\n}\n\nint main(int argc, char **argv) {\n    /* --- defaults --- */\n    int N          = 256;\n    int steps      = 1000;\n    int snapEvery  = 50;\n    double alpha   = 0.10;      /* diffusion coefficient */\n    const char *outDir = "./out";\n\n    /* --- arg parse (tiny, positional-by-flag) --- */\n    for (int a = 1; a < argc; ++a) {\n        if      (!strcmp(argv[a], "--N")         && a+1 < argc) N         = atoi(argv[++a]);\n        else if (!strcmp(argv[a], "--steps")     && a+1 < argc) steps     = atoi(argv[++a]);\n        else if (!strcmp(argv[a], "--snapEvery") && a+1 < argc) snapEvery = atoi(argv[++a]);\n        else if (!strcmp(argv[a], "--alpha")     && a+1 < argc) alpha     = atof(argv[++a]);\n        else if (!strcmp(argv[a], "--outDir")    && a+1 < argc) outDir    = argv[++a];\n        else { fprintf(stderr, "unknown arg: %s\\n", argv[a]); return 1; }\n    }\n    if (N < 8) { fprintf(stderr, "N must be >= 8\\n"); return 1; }\n\n    /* Spatial and temporal step. Timestep is bounded by CFL for explicit    */\n    /* Euler on a 5-point stencil: dt < h^2 / (4*alpha). We take a safety    */\n    /* factor of 4 so students who tune alpha up a little don\'t blow up.     */\n    const double h  = 1.0 / (double)N;\n    const double dt = 0.25 * h * h / alpha;   /* CFL-safe */\n\n    /* Allocate two grids, ping-pong. Contiguous, row-major, C-order. */\n    double *u    = calloc((size_t)N * N, sizeof(double));\n    double *unew = calloc((size_t)N * N, sizeof(double));\n    if (!u || !unew) { fprintf(stderr, "alloc failed\\n"); return 1; }\n    initInitials(u, N);\n\n    /* --- main loop --- */\n    double t0 = wallSeconds();\n    double ioSec = 0.0;\n    int    frames = 0;\n\n    /* Initial snapshot */\n    char path[256];\n    double ta = wallSeconds();\n    snprintf(path, sizeof path, "%s/frame_%04d.npy", outDir, frames++);\n    writeNpy2D(path, u, N);\n    ioSec += wallSeconds() - ta;\n\n    /* Science-check header */\n    char sciPath[256];\n    snprintf(sciPath, sizeof sciPath, "%s/science.csv", outDir);\n    FILE *sci = fopen(sciPath, "w");\n    if (!sci) { perror(sciPath); return 1; }\n    fprintf(sci, "step,time,sum_u\\n");\n    fprintf(sci, "0,0.0,%.15e\\n", gridSum(u, N));\n\n    for (int s = 1; s <= steps; ++s) {\n        /* -----------------------------------------------------------------\n         *  STENCIL - THIS IS WHAT YOU (THE STUDENT) FILL IN.\n         *\n         *  For each interior cell (i, j), the new value is the old value plus\n         *  the diffusion coefficient (alpha) times the timestep (dt) times\n         *  the discrete Laplacian.\n         *\n         *  The discrete Laplacian on a uniform grid with spacing h is\n         *      (u[i-1,j] + u[i+1,j] + u[i,j-1] + u[i,j+1] - 4*u[i,j]) / (h*h)\n         *\n         *  With PERIODIC boundaries, "i-1" at the edge wraps to N-1, and\n         *  "i+1" at the edge wraps to 0. Use modular arithmetic: the ternary\n         *      (i == 0 ? N-1 : i-1)\n         *  is one clean way, or you can compute (i + N - 1) % N.\n         *\n         *  Store the result in unew[i*N + j]. Row-major indexing.\n         * ----------------------------------------------------------------- */\n        for (int i = 0; i < N; ++i) {\n            int ip = (i + 1) % N;\n            int im = (i + N - 1) % N;\n            for (int j = 0; j < N; ++j) {\n                int jp = (j + 1) % N;\n                int jm = (j + N - 1) % N;\n\n                /* TODO(student): replace this line with the 5-point update. */\n                /* HINT: use u[im*N + j], u[ip*N + j], u[i*N + jm], u[i*N + jp], u[i*N + j]  */\n                unew[i*N + j] = u[i*N + j];   /* <-- REPLACE ME */\n            }\n        }\n\n        /* swap ping-pong buffers */\n        double *tmp = u; u = unew; unew = tmp;\n\n        /* periodic snapshot + science-check row */\n        if ((s % snapEvery) == 0 || s == steps) {\n            ta = wallSeconds();\n            snprintf(path, sizeof path, "%s/frame_%04d.npy", outDir, frames++);\n            writeNpy2D(path, u, N);\n            ioSec += wallSeconds() - ta;\n            fprintf(sci, "%d,%.6f,%.15e\\n", s, s*dt, gridSum(u, N));\n            printf("  step %6d  t=%.4f  sum(u)=%.6e  frames=%d\\n",\n                   s, s*dt, gridSum(u, N), frames);\n        }\n    }\n    fclose(sci);\n\n    double wall = wallSeconds() - t0;\n    /* MLUP/s: million lattice updates per second (a standard stencil metric). */\n    double mlups = ((double)N * N * (double)steps) / wall / 1.0e6;\n\n    /* Append (or create) timings.csv with the schema every later lab shares. */\n    char tPath[256]; snprintf(tPath, sizeof tPath, "%s/timings.csv", outDir);\n    int newFile = access(tPath, F_OK) != 0;\n    FILE *tf = fopen(tPath, "a");\n    if (!tf) { perror(tPath); return 1; }\n    if (newFile) fprintf(tf, "lab,variant,N,steps,threads,ranks,wall_s,mlups,io_s\\n");\n    fprintf(tf, "01,serial,%d,%d,1,1,%.6f,%.3f,%.6f\\n", N, steps, wall, mlups, ioSec);\n    fclose(tf);\n\n    printf("\\nDONE  N=%d  steps=%d  wall=%.3fs  MLUP/s=%.2f  io=%.3fs (%.1f%% of wall)  frames=%d\\n",\n           N, steps, wall, mlups, ioSec, 100.0 * ioSec / wall, frames);\n\n    free(u); free(unew);\n    return 0;\n}\n'
labDir = pathlib.Path(env['labDir'])
(labDir / 'heat2D.c').write_text(heat2DC)
showFile(labDir / 'heat2D.c', language='c', maxLines=60, title='heat2D.c (first 60 lines)')


**[Notebook cell]** Open `~/lab01/heat2D.c` in JupyterLab's file browser, find the comment block that says **`STENCIL - THIS IS WHAT YOU (THE STUDENT) FILL IN`**, and replace the marked `TODO` line with the update from Part 1b. Save the file. The next cell will scp it to Crux and try to compile it — if the compile fails or the conservation check drifts, you'll see it right away.

> **💡 Sanity check on your line.** With periodic boundaries, `sum(u)` over the whole > grid should stay constant to about 1e-10 for the entire run. Diffusion moves heat > around but does not create or destroy it. If Part 3's checkpoint shows the sum > drifting away from its initial value, your stencil is unbalanced — go back and check > the sign of `- 4.0 * u[i*N + j]` and the division by `(h*h)`.


In [ ]:
# [Hub] Confirm the student edited the TODO line (a tiny lexical check).
# This does NOT verify correctness - only that the placeholder line is gone.
src = (labDir / 'heat2D.c').read_text()
placeholder = 'unew[i*N + j] = u[i*N + j];   /* <-- REPLACE ME */'
if placeholder in src:
    showNote('You have not edited the stencil line yet. Open ~/lab01/heat2D.c in the file browser, '
             'find the TODO block, and replace the marked line with the 5-point update from Part 1b.',
             kind='warn', title='heat2D.c still contains the placeholder')
else:
    showNote('Placeholder line replaced. Now go compile and run it.', kind='ok')


In [ ]:
checkpoint("Part 2 - stencil filled in", [
    check("heat2D.c on the Hub", fileExists(str(labDir / 'heat2D.c'))),
    check("stencil placeholder removed",
          lambda: ('<-- REPLACE ME' not in (labDir / 'heat2D.c').read_text(),
                   'placeholder gone' if '<-- REPLACE ME' not in (labDir / 'heat2D.c').read_text()
                   else 'placeholder still there'),
          hint='Edit ~/lab01/heat2D.c, replace the TODO line with the 5-point update from Part 1b.'),
])


## Part 3 · Write your own PBS script, then build and run

Lab 00 handed you a PBS script written by the notebook. This time **you write it**. Everything you need is on the [Crux Running Jobs page](https://docs.alcf.anl.gov/crux/queueing-and-running-jobs/running-jobs/); the required directives for CS 455 are:

```
#PBS -N lab01Heat            # any job name
#PBS -A UIC-CS455-Sp2027     # the class project
#PBS -q debug                # the small, fast-turnaround queue
#PBS -l select=1:system=crux # one Crux compute node
#PBS -l walltime=00:10:00    # <= 10 min; debug queue enforces this
#PBS -l filesystems=home:eagle   # required at ALCF, see lab 00 Part 2
#PBS -j oe                   # merge stderr into stdout
```

The body of the script has to (a) `cd` into the lab directory on `/eagle`, (b) load the compiler module, (c) build `heat2D.c`, and (d) run `./heat2D` with the arguments you want.

The next cell writes a **starter** version — you'll edit it in a moment to fill in the arguments and, if you want, swap the programming environment.


In [ ]:
# [Hub] Write the starter PBS script. You WILL edit this file in the next step.
pbsScript = f'''\
#!/bin/bash
#PBS -N lab01Heat
#PBS -A {env['HPC_PROJECT']}
#PBS -q {env.get('HPC_QUEUE','debug')}
#PBS -l select=1:system=crux
#PBS -l walltime=00:10:00
#PBS -l filesystems=home:eagle
#PBS -j oe
#PBS -o {env['HPC_LAB_DIR']}/heat.out

# --- your job body starts here -----------------------------------------
cd {env['HPC_LAB_DIR']}
mkdir -p out

# Use the default Cray PE compiler wrapper 'cc'. If you prefer GCC, uncomment:
#   module swap PrgEnv-cray PrgEnv-gnu
module list 2>&1 | head -5

# Build. -O3 for optimization, -Wall to see any warnings you introduced.
cc -O3 -Wall -o heat2D heat2D.c -lm

# Run. You can edit these arguments after your first successful run.
./heat2D --N 256 --steps 1000 --snapEvery 100 --outDir ./out
'''
(labDir / 'heat2D.pbs').write_text(pbsScript)
showFile(labDir / 'heat2D.pbs', language='bash')


**[Notebook cell]** Now ship both files to Crux and submit. If your stencil is right, the job takes about 5-15 seconds of compute time (once it's off the queue). The debug queue on Crux usually schedules within a couple of minutes.


In [ ]:
# [Hub -> Crux] Copy heat2D.c and heat2D.pbs to your lab dir on /eagle.
sshPut(str(labDir / 'heat2D.c'),   env['HPC_LAB_DIR'] + '/heat2D.c')
sshPut(str(labDir / 'heat2D.pbs'), env['HPC_LAB_DIR'] + '/heat2D.pbs')
# Submit via the helper (still qsub under the hood - see lab 00 Part 7 under-the-hood).
jobID = submitJob(env['HPC_LAB_DIR'] + '/heat2D.pbs')
print('job id:', jobID)


In [ ]:
# [Hub] Wait for the job to leave the queue. Debug queue is usually 1-3 minutes.
waitJob(jobID, pollSeconds=15, maxSeconds=900)


In [ ]:
# [Hub -> Crux] Pull the output back. sshGet -r brings the whole out/ folder,
# including all frame_XXXX.npy files, timings.csv, science.csv, and heat.out.
sshGet(env['HPC_LAB_DIR'] + '/out',      str(labDir / 'out'))
sshGet(env['HPC_LAB_DIR'] + '/heat.out', str(labDir / 'heat.out'))
showFile(labDir / 'heat.out', language='text', maxLines=25, title='heat.out (job stdout)')


In [ ]:
checkpoint("Part 3 - first successful run", [
    check("heat.out was produced", fileExists(str(labDir / 'heat.out'))),
    check("job printed a DONE line", fileContains(str(labDir / 'heat.out'), 'DONE  N=')),
    check("timings.csv exists", fileExists(str(labDir / 'out' / 'timings.csv'))),
    check("at least a few frames were written",
          lambda: (len(list((labDir/'out').glob('frame_*.npy'))) >= 5,
                   f"{len(list((labDir/'out').glob('frame_*.npy')))} frames")),
    check("conservation check exists", fileExists(str(labDir / 'out' / 'science.csv'))),
])


## Part 4 · Your first animation and your first measurement row

Now that the job ran, do two things with the output. First, watch the physics — turn the sequence of `.npy` frames into an MP4. Second, look at the measurement CSVs and understand what you just captured.


### 4a · Animate the field

`renderField` from `labHelpers.py` reads a directory of frames and writes a video. It decides where to render based on file size (see lab 11 for the in-situ story); for lab 01 with `N=256` all frames fit easily in memory on the Hub.


In [ ]:
# [Hub] Assemble the frames into an MP4 (falls back to GIF if ffmpeg is missing).
ensureDependencies(['numpy', 'matplotlib', 'imageio', 'imageio-ffmpeg'])
import numpy as np, matplotlib.pyplot as plt, imageio.v2 as iio
frames = sorted((labDir / 'out').glob('frame_*.npy'))
print(f'found {len(frames)} frames')
images = []
vmin = 0.10; vmax = 1.00
for fp in frames:
    arr = np.load(fp)
    fig, ax = plt.subplots(figsize=(4,4), dpi=100)
    ax.imshow(arr, cmap='inferno', vmin=vmin, vmax=vmax, origin='lower')
    ax.set_axis_off()
    fig.tight_layout(pad=0)
    fig.canvas.draw()
    rgba = np.asarray(fig.canvas.buffer_rgba())
    images.append(rgba[:, :, :3].copy())
    plt.close(fig)
outVideo = labDir / 'initials.mp4'
try:
    iio.mimwrite(str(outVideo), images, fps=12, macro_block_size=1)
    print(f'wrote {outVideo}')
except Exception as e:
    outVideo = labDir / 'initials.gif'
    iio.mimwrite(str(outVideo), images, duration=1/12)
    print(f'ffmpeg not available, wrote GIF instead: {outVideo}')
from IPython.display import Image, Video, display
display(Video(str(outVideo), embed=False) if outVideo.suffix=='.mp4' else Image(str(outVideo)))


**Try it:** open `~/lab01/heat2D.c` and change the `initInitials` function to spell YOUR initials — pick two 8x8 hot regions at coordinates that spell letters you recognize. Re-run Part 3 (the ship-submit-fetch cells) and this cell, and you have your own personal artifact for the lab.


### 4b · Look at what got measured

Three CSVs came back from Crux. Each answers one question about the run.


In [ ]:
# [Hub] Load the measurement CSVs and show them.
ensureDependencies(['pandas'])
import pandas as pd
timings = pd.read_csv(labDir / 'out' / 'timings.csv')
science = pd.read_csv(labDir / 'out' / 'science.csv')
print('=== timings.csv (performance and I/O) ===')
print(timings.to_string(index=False))
print()
print('=== science.csv (conservation check, first + last 5 rows) ===')
print(pd.concat([science.head(5), science.tail(5)]).to_string(index=False))
print()
drift = science['sum_u'].iloc[-1] - science['sum_u'].iloc[0]
rel   = abs(drift) / abs(science['sum_u'].iloc[0])
print(f'sum(u) drift: {drift:+.3e} absolute, {rel:.2e} relative')
if rel > 1e-8:
    showNote(f'Sum-of-u drifted by {rel:.1e} - larger than roundoff (~1e-14 to 1e-10). '
             'Your stencil is probably missing a term or has a sign error. Go back to Part 1b '
             'and check that the update includes -4*u[i,j] and divides by h*h.',
             kind='warn', title='Conservation check failed')
else:
    showNote(f'Conservation check passed: relative drift {rel:.1e}, at the level of floating-point '
             'roundoff. Your stencil is balanced.', kind='ok')


In [ ]:
checkpoint("Part 4 - measured and animated", [
    check("animation was written",
          lambda: (any((labDir/n).exists() for n in ('initials.mp4','initials.gif')),
                   'initials.mp4' if (labDir/'initials.mp4').exists() else 'initials.gif')),
    check("timings.csv has one row",
          lambda: (len(pd.read_csv(labDir/'out'/'timings.csv')) == 1,
                   f"{len(pd.read_csv(labDir/'out'/'timings.csv'))} row(s)")),
    check("conservation drift < 1e-8",
          lambda: (
              (lambda s: (abs(s['sum_u'].iloc[-1]-s['sum_u'].iloc[0])/abs(s['sum_u'].iloc[0]) < 1e-8,
                          f"drift={abs(s['sum_u'].iloc[-1]-s['sum_u'].iloc[0])/abs(s['sum_u'].iloc[0]):.2e}"))(
                pd.read_csv(labDir/'out'/'science.csv'))),
          hint='Sum(u) should stay constant to ~1e-10 on a periodic domain. If it drifts, your '
               'stencil has a sign error or missing term - go back to Part 1b.'),
])


## Part 5 · Four figures · performance, I/O, science, snapshots

One run gives you one point on any performance plot. But you can already produce **four publication-quality figures** that will grow through the rest of the semester as later labs add rows to the same CSVs.


In [ ]:
# [Hub] Publication house style - same across every lab from here on.
palette = applyHouseStyle()


### 5a · Performance strip · MLUP/s vs grid size

[MLUP/s](https://www.top500.org/) — millions of lattice updates per second — is the canonical stencil-code metric. Higher is better. One point today; lab 03 (OpenMP) and lab 06 (MPI) will add columns for parallel variants that this baseline is the reference for.


In [ ]:
# [Hub] Even one point is a valid figure - it's the baseline mark that grows over time.
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.plot(timings['N'], timings['mlups'], marker='o', linestyle='-', label='serial')
ax.set_xlabel('grid size N (points per side)')
ax.set_ylabel('performance (MLUP/s)')
ax.set_title('Serial heat-stencil throughput')
ax.legend()
saveFigure(fig, 'performance_baseline', figuresDir=str(labDir/'figures'))


### 5b · I/O strip · how much of your wall time was spent writing frames?

This is the most-often-overlooked HPC number. A run that spends 40% of its wall time in I/O will not benefit from adding more compute — you have an I/O bottleneck, and lab 13 (parallel I/O) is where you'll fix it. For today: just look.


In [ ]:
# [Hub] I/O as a fraction of wall time.
fig, ax = plt.subplots()
ioFrac = 100.0 * timings['io_s'] / timings['wall_s']
ax.bar(['I/O', 'compute'], [ioFrac.iloc[0], 100 - ioFrac.iloc[0]])
ax.set_ylabel('% of wall time')
ax.set_title(f"I/O share of wall time (N={int(timings['N'].iloc[0])}, snapEvery=100)")
saveFigure(fig, 'io_share', figuresDir=str(labDir/'figures'))


### 5c · Science conservation check · sum(u) vs time

On a periodic domain diffusion moves heat around but does not create or destroy it, so the total heat (sum of $u$) must stay constant to roundoff. This plot is a straight line if your stencil is correct.


In [ ]:
# [Hub] Conservation over time.
fig, ax = plt.subplots()
ax.plot(science['time'], science['sum_u'], marker='.', linestyle='-')
ax.set_xlabel('simulation time t')
ax.set_ylabel(r'$\sum u$ (should be flat)')
ax.set_title('Heat conservation (periodic BCs)')
ax.ticklabel_format(useOffset=False)
saveFigure(fig, 'conservation', figuresDir=str(labDir/'figures'))


### 5d · Field snapshots · watch it dissipate

Same physics as the animation, but pinned in a 4-panel figure you can drop into a lab report or presentation.


In [ ]:
# [Hub] 4-panel snapshot at t = 0, T/4, T/2, T.
import numpy as np
picks = [frames[0], frames[len(frames)//4], frames[len(frames)//2], frames[-1]]
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, fp in zip(axes, picks):
    arr = np.load(fp)
    im = ax.imshow(arr, cmap='inferno', vmin=vmin, vmax=vmax, origin='lower')
    frameIdx = int(fp.stem.split('_')[1])
    ax.set_title(f'frame {frameIdx}')
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(im, ax=axes, shrink=0.8, label='u')
saveFigure(fig, 'snapshots_4panel', figuresDir=str(labDir/'figures'))


In [ ]:
checkpoint("Part 5 - four figures produced", [
    check("performance figure written",
          fileExists(str(labDir / 'figures' / 'performance_baseline.pdf'))),
    check("I/O share figure written",
          fileExists(str(labDir / 'figures' / 'io_share.pdf'))),
    check("conservation figure written",
          fileExists(str(labDir / 'figures' / 'conservation.pdf'))),
    check("snapshot 4-panel written",
          fileExists(str(labDir / 'figures' / 'snapshots_4panel.pdf'))),
])


## Part 6 · Under the hood · what you wrote yourself this time

Lab 00 revealed what the `labHelpers` functions do. Lab 01 revealed a different layer — **the things you wrote yourself, not the helper.** Three files are worth reading back:

1. **`heat2D.pbs`** — every `#PBS` directive you now use is documented on the [Crux Running Jobs page](https://docs.alcf.anl.gov/crux/queueing-and-running-jobs/running-jobs/). You will copy-and-adapt this script for the next 6 labs; changes will be adding `-l select=N:system=crux` for multi-node MPI runs and `-l place=scatter` for affinity.
2. **`heat2D.c` timer** — `clock_gettime(CLOCK_MONOTONIC, ...)` is the right way to measure wall time on Linux. Never use `time()` for HPC timing (1-second granularity) or `clock()` (measures CPU time, not wall time). Lab 02 goes deeper on this.
3. **`.npy` writer** — 25 lines of C write a file numpy can load with a single `np.load(path)` — no dependency on HDF5 or any other library. See the [`.npy` format spec](https://numpy.org/doc/stable/reference/generated/numpy.lib.format.html).

**Next lab** (lab 02, PerformanceMeasurement) makes the measurement itself the subject: wall vs CPU time, `perf stat` counters, the [roofline model](https://en.wikipedia.org/wiki/Roofline_model), why one number lies, and — as we said in Part 4 — power measurement.

**After that** (lab 03, OpenMP), you'll add a `#pragma omp parallel for` above the outer stencil loop, submit with `OMP_NUM_THREADS=1,2,4,8,16,32,64`, and this lab's single MLUP/s point becomes the leftmost point on your first strong-scaling curve. Every serial number you produced today is a reference the parallel version has to beat.


## Wrap up

You have a working serial baseline you understand top to bottom, a personal animation of your initials dissipating, and the first row of the `timings.csv` schema every later lab shares. From this point forward, every lab in the sequence produces numbers that plug into figures you already know how to make.


### Lab scorecard


In [ ]:
labSummary("Serial Baseline")


---
### One-minute feedback

What worked, what didn't, what should be clearer. Anonymous to your classmates; goes straight to the instructor.


In [ ]:
feedback("Serial Baseline")
